# **NLP701@MBZUAI Fall 2024 - Lab 10**
- Location: Lab 1, 2, 3, 4
- Time: 11:00 - 13:00, October  30, 2024


## **Learning Outcomes**
- Critically analyze, evaluate, and improve the performance of RNN and LSTM.
- Gain proficiency in implementing neural networks.

## **Learning Activities**
- Implement an NER tagger using RNNs.
- Improving the model by adding character level information or other methods.


# **RNNs for Sequence Labeling**
In this lab exercise, we build an NER tagger for Arabic using **RNNs**.
We use the same dataset as assignment 1, i.e., ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).

## **Data Preparation**

In [1]:
!pip install scikit-learn seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16252 sha256=cd7da6b64d7e8917573d4704e788385463fd8a018a45bba14d62b692d7a36909
  Stored in directory: /home/khang.nhat/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [2]:
!wget "https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip"
!unzip ANERcorp-CamelLabSplits.zip

--2026-03-28 07:35:05--  https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
Resolving camel.abudhabi.nyu.edu (camel.abudhabi.nyu.edu)... 91.230.41.24
Connecting to camel.abudhabi.nyu.edu (camel.abudhabi.nyu.edu)|91.230.41.24|:443... connected.
HTTP request sent, awaiting response... 200 
Length: 926414 (905K) [application/zip]
Saving to: ‘ANERcorp-CamelLabSplits.zip’

ANERcorp-CamelLabSp 100%[===================>] 904.70K  --.-KB/s    in 0.1s    

2026-03-28 07:35:07 (7.28 MB/s) - ‘ANERcorp-CamelLabSplits.zip’ saved [926414/926414]

Archive:  ANERcorp-CamelLabSplits.zip
   creating: ANERcorp-CamelLabSplits/
  inflating: __MACOSX/._ANERcorp-CamelLabSplits  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_Benajiba.txt  
  inflating: __MACOSX/ANERcorp-CamelLabSplits/._ANERCorp_Benajiba.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt  
  inflating: __MACOSX/ANERcorp-CamelLabSplits/._ANERCorp_CamelLab_train.txt  
  inflating: ANERcorp-CamelLabSplits/REA

In [3]:
from sklearn.model_selection import train_test_split

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

In [4]:
# check the dataset
print('Size of train_sents: %d'%len(train_sents))
print('Size of dev_sents: %d'%len(dev_sents))

print('The training data looks like:')
for sent in train_sents[:5]:
  print(sent)

Size of train_sents: 3575
Size of dev_sents: 398
The training data looks like:
[('وقال', 'O'), ('إن', 'O'), ('العملية', 'O'), ('استهدفت', 'O'), ('"', 'O'), ('بنى', 'O'), ('تحتية', 'O'), ('تستعمل', 'O'), ('لتخزين', 'O'), ('أسلحة', 'O'), ('عائدة', 'O'), ('للجهاد', 'O'), ('الإسلامي', 'O'), ('"', 'O'), ('،', 'O'), ('واوضح', 'O'), ('إن', 'O'), ('إسرائيل', 'B-LOC'), ('أخطرت', 'O'), ('سكان', 'O'), ('المنطقة', 'O'), ('بوقوع', 'O'), ('الغارة', 'O'), ('.', 'O')]
[('وهذه', 'O'), ('هي', 'O'), ('المباراة', 'O'), ('الرسمية', 'O'), ('الأولى', 'O'), ('لمنتخب', 'O'), ('إنجلترا', 'B-LOC'), ('تحت', 'O'), ('قيادة', 'O'), ('مدربه', 'O'), ('ستيف', 'B-PER'), ('مكلارين', 'I-PER'), ('الذي', 'O'), ('خلف', 'O'), ('السويدي', 'O'), ('غوران', 'B-PER'), ('إريكسون', 'I-PER'), ('بعد', 'O'), ('نهائيات', 'O'), ('المونديال', 'B-MISC'), ('.', 'O')]
[('في', 'O'), ('عيد', 'O'), ('الملكة', 'O'), ('السماوية', 'O'), ('"', 'O'), ('تيني', 'B-PER'), ('هاو', 'I-PER'), ('"', 'O'), ('يسود', 'O'), ('إحساس', 'O'), ('بالروحانية', 'O'),

## **Build Vocabulary**

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

torch.manual_seed(1)

def build_vocab(sents, idx, special_tokens=['UNK']):
    vocab = {}
    if special_tokens:
        vocab = {k: v for v, k in enumerate(special_tokens)}

    for sent in sents:
        for word in sent:
            if word[idx] not in vocab:
                vocab[word[idx]] = len(vocab)

    return vocab

def prepare_sequence(seq, to_idx):
    idxs = []
    for w in seq:
        if w in to_idx:
            idxs.append(to_idx[w])
        else:
            idxs.append(to_idx['UNK'])
    return torch.tensor(idxs, dtype=torch.long)

In [6]:
word_to_idx = build_vocab(train_sents, 0)
tag_to_idx = build_vocab(train_sents, 1, special_tokens=None)

## **Create the model**

In [7]:
class RNNTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(RNNTagger, self).__init__()
        self.hidden_dim = hidden_dim
        # Word embedding layer. This maps each word to a vector representation.
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The RNN takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        # You can utilize the multi-layer rnn by changing num_layers.
        # Also you can use the bidirectional rnn with bidirectional=True.
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=1, bidirectional=False)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        rnn_out, _ = self.rnn(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(rnn_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

    def predict(self, sentence):
        with torch.no_grad():
            inputs = prepare_sequence(sentence, word_to_idx)
            tag_scores = self.forward(inputs)
            _, indices = torch.max(tag_scores, 1)
            tags = []
            for i in range(len(indices)):
                for key, value in tag_to_idx.items():
                    if indices[i] == value:
                        tags.append(key)
        return tags

## **Train the model**

In [8]:
def train(model, train_sents, lr=0.1, epochs=3):
    loss_function = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for epoch in range(epochs):

        for sentence in tqdm(train_sents):
            # Step 1. Remember that Pytorch accumulates gradients.
            # We need to clear them out before each instance
            optimizer.zero_grad() # or model.zero_grad()

            # Step 2. Get our inputs ready for the network, that is, turn them into
            # Tensors of word indices.
            x = [word[0] for word in sentence]
            y = [word[1] for word in sentence]
            sentence_in = prepare_sequence(x, word_to_idx)
            targets = prepare_sequence(y, tag_to_idx)

            # Step 3. Run our forward pass.
            tag_scores = model(sentence_in)

            # Step 4. Compute the loss, gradients, and update the parameters by
            #  calling optimizer.step()
            loss = loss_function(tag_scores, targets)
            loss.backward()
            optimizer.step()

        print(f'epoch: {epoch+1}, loss: {loss}')

    return model

In [9]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

RNN_model = RNNTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_idx), len(tag_to_idx))
RNN_tagger = train(RNN_model, train_sents, epochs=5)

torch.save(RNN_tagger.state_dict(), './rnn_tagger.pt')

  0%|          | 0/3575 [00:00<?, ?it/s]/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
100%|██████████| 3575/3575 [00:13<00:00, 260.41it/s]


epoch: 1, loss: 0.1423053741455078


100%|██████████| 3575/3575 [00:08<00:00, 402.38it/s]


epoch: 2, loss: 0.0937434509396553


100%|██████████| 3575/3575 [00:12<00:00, 284.42it/s]


epoch: 3, loss: 0.055363357067108154


100%|██████████| 3575/3575 [00:09<00:00, 369.78it/s]


epoch: 4, loss: 0.03216924890875816


100%|██████████| 3575/3575 [00:08<00:00, 421.94it/s]

epoch: 5, loss: 0.029427779838442802


## **Evaluate the Model**

Load the model:

In [10]:
RNN_tagger = RNNTagger(EMBEDDING_DIM,
                       HIDDEN_DIM,
                       len(word_to_idx),
                       len(tag_to_idx))
RNN_tagger.load_state_dict(torch.load('./rnn_tagger.pt'))
RNN_tagger.eval()

RNNTagger(
  (word_embeddings): Embedding(27364, 128)
  (rnn): RNN(128, 256)
  (hidden2tag): Linear(in_features=256, out_features=9, bias=True)
)

Evalute the model:

In [11]:
from seqeval.metrics import f1_score

dev = [[word[0] for word in sent] for sent in dev_sents]
dev_gold = [[word[1] for word in sent] for sent in dev_sents]

dev_pred = [RNN_tagger.predict(sent) for sent in dev]
f1_score(dev_gold, dev_pred)

0.44234404536862004

# **Exercise 01: LSTM**

Create the model:

In [12]:
class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        # Accept input as [seq_len] or [batch, seq_len]
        if sentence.dim() == 1:
            sentence = sentence.unsqueeze(0)
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds)
        tag_space = self.hidden2tag(lstm_out)
        return tag_space

    def predict(self, sentence):
        # sentence: list of word indices (ints)
        with torch.no_grad():
            inputs = torch.tensor(sentence, dtype=torch.long).unsqueeze(0)  # batch size 1
            tag_scores = self.forward(inputs)
            predicted_indices = torch.argmax(tag_scores, dim=2).squeeze(0).tolist()
        return [list(tag_to_idx.keys())[list(tag_to_idx.values()).index(idx)] for idx in predicted_indices]

Train the model:

In [13]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

# Build or reuse vocabs
word_to_idx = build_vocab(train_sents, 0)
tag_to_idx = build_vocab(train_sents, 1, special_tokens=None)

# Helper: sentences to indices
def sents_to_idx(sents, w2i):
    return [[w2i.get(w, w2i.get('UNK', 0)) for w in [tok for tok, _ in sent]] for sent in sents]

def sents_tags_to_idx(sents, t2i):
    return [[t2i[tag] for _, tag in sent] for sent in sents]

train_X = sents_to_idx(train_sents, word_to_idx)
train_y = sents_tags_to_idx(train_sents, tag_to_idx)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

lstm_model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_idx), len(tag_to_idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

EPOCHS = 5
for epoch in range(EPOCHS):
    total_loss = 0.0
    for sent_idx, tag_idx in tqdm(zip(train_X, train_y), total=len(train_X)):
        inputs = torch.tensor(sent_idx, dtype=torch.long, device=device)
        targets = torch.tensor(tag_idx, dtype=torch.long, device=device)

        optimizer.zero_grad()
        logits = lstm_model(inputs)             # [1, seq_len, num_tags]
        logits = logits.squeeze(0)              # [seq_len, num_tags]
        loss = criterion(logits, targets)       # token-level CE
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {total_loss/len(train_X):.4f}")

# Save LSTM model
torch.save(lstm_model.state_dict(), './lstm_tagger.pt')


  0%|          | 16/3575 [00:00<00:22, 158.14it/s]

100%|██████████| 3575/3575 [00:11<00:00, 302.84it/s]


Epoch 1/5 - loss: 0.3960


100%|██████████| 3575/3575 [00:10<00:00, 328.76it/s]


Epoch 2/5 - loss: 0.1783


...

Epoch 3/5 - loss: 0.0738


100%|██████████| 3575/3575 [00:11<00:00, 317.01it/s]


Epoch 4/5 - loss: 0.0244


100%|██████████| 3575/3575 [00:11<00:00, 324.48it/s]


Epoch 5/5 - loss: 0.0114


Evalute the model:

In [14]:
from seqeval.metrics import f1_score, classification_report

# Prepare dev data
dev_words = [[w for w, _ in sent] for sent in dev_sents]
dev_gold = [[t for _, t in sent] for sent in dev_sents]

# Map words to indices for prediction
lstm_model.eval()
dev_pred = []
with torch.no_grad():
    for sent in dev_words:
        idxs = [word_to_idx.get(w, word_to_idx.get('UNK', 0)) for w in sent]
        logits = lstm_model(torch.tensor(idxs, dtype=torch.long, device=device))
        pred_idx = torch.argmax(logits.squeeze(0), dim=1).tolist()
        inv_tag = {v: k for k, v in tag_to_idx.items()}
        dev_pred.append([inv_tag[i] for i in pred_idx])

print('LSTM Dev F1:', f1_score(dev_gold, dev_pred))
print(classification_report(dev_gold, dev_pred))

LSTM Dev F1: 0.6573660714285715
              precision    recall  f1-score   support

         LOC       0.81      0.77      0.79       397
        MISC       0.65      0.49      0.56        87
         ORG       0.62      0.54      0.58       175
         PER       0.49      0.60      0.54       241

   micro avg       0.66      0.65      0.66       900
   macro avg       0.64      0.60      0.62       900
weighted avg       0.67      0.65      0.66       900



## **What would be a good example to show that LSTM performs better than a simple RNN?**
- No coding involved, and no need to use Arabic as an example.
- Imagine you're writing a paper, and you want to show an example that works well in LSTM compared to RNN.
- What kind of example would you present and why?

In [15]:
#[You idea]

# A good example to show that LSTM performs better than a simple RNN is a sentence with long-range dependencies.
# For instance, consider the sentence:
# "The book that the professor who the students liked recommended was very interesting."
# In this sentence, the word "was" refers to the subject "book", which is far from the verb due to intervening clauses.
# A simple RNN may struggle to remember the subject "book" when predicting the tag for "was" because of the long distance,
# while an LSTM, with its gating mechanisms, can better capture such long-range dependencies and correctly tag "was" as a verb.
# This demonstrates the LSTM's advantage in handling sequences where important information is separated by many tokens.


# **Bonus Exercise: How Would You Improve the Model?**
- What can be done on top of the simple LSTM tagger to improve the performance?

In [16]:
#[Your Code]
# There are several ways to improve the simple LSTM tagger:
# 1. Use a bidirectional LSTM to capture both past and future context.
# 2. Add more layers (stacked LSTMs) for deeper representations.
# 3. Use pre-trained word embeddings (like GloVe or FastText) instead of training from scratch.
# 4. Add character-level embeddings using a CNN or another LSTM to capture subword information.
# 5. Apply dropout for regularization to prevent overfitting.
# 6. Use a CRF (Conditional Random Field) layer on top of the LSTM to model tag dependencies.
# 7. Incorporate additional features, such as POS tags, capitalization, or affixes.
# 8. Tune hyperparameters such as learning rate, batch size, and hidden size.
# 9. Use layer normalization or batch normalization.
# 10. Try more advanced architectures like Transformers.

# Example: Using a bidirectional LSTM and pre-trained embeddings
# (Pseudocode, as actual implementation would require model redefinition)
#
# class ImprovedTagger(nn.Module):
#     def __init__(self, ...):
#         ...
#         self.word_embeddings = nn.Embedding.from_pretrained(pretrained_weights, freeze=False)
#         self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, 
#                             bidirectional=True, dropout=0.5)
#         self.hidden2tag = nn.Linear(hidden_dim * 2, tagset_size)
#         ...
#     def forward(self, ...):
#         ...
#         return tag_scores


## **Reference**
https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html